# Construction of the generator matrix representation — demo

Minimal worked example. All logic lives in the `poly_jump_signature` package (see `README.md`).



For a polynomial jump-diffusion, the expected signature truncated at level $N$ is obtained in closed form as

$$\mathbb{E}\big[\mathbb{X}_T^{\leq N}\big]=
 \exp(TL^\top)\,\mathbb{X}_0^{\leq N} = \exp(TL^\top)\,\epsilon_\varnothing$$

i.e. the row of $\exp(TL)$ indexed by the empty word $\varnothing$. Everything therefore hinges on **assembling $L$**, the matrix representing the generator's action on words — which is what this notebook demonstrates. Once $L$ is available, `expected_signature` is a single call to `scipy.linalg.expm`.

### What this notebook does

It walks through the whole construction of the matrix $L$ end to end on the smallest non-trivial case, $d = 2$ and $\text{max\_rank} = 2$:

1. build the coefficient table with `create_storage_df`,
2. export it to CSV, fill it in, reload it with `load_storage_from_template_df`,
3. pack it into the flat tensor basis with `derivatives_to_tensor_df`,
4. assemble $L$ with `matrix_generator`.

**The coefficient values used here are arbitrary**, chosen only to exercise the code paths — they do not come from any particular model. For $L$ built from a genuine set of characteristics, see `expected_signature_poly_jump_diffusion.ipynb` (time-extended OU with compound Poisson jumps), where the resulting expected signature is checked against closed-form moments.

In [1]:
import sys
if ".." not in sys.path:
    sys.path.insert(0, "..")

from poly_jump_signature import (
    create_storage_df, export_template_df, load_storage_from_template_df,
    derivatives_to_tensor_df, matrix_generator, expected_signature,
    fromlentonum, fromnumtoword, fromwordtonum,
)
import numpy as np

## 0. The flat word basis

Everything below is indexed by **words** over the alphabet $\{1, \dots, d\}$. A word $I = (i_1, \dots, i_{|I|})$ labels a signature component $\langle \epsilon_I,\mathbb{X}\rangle$, and `words.py` sets up a bijection between words and integers:

- `fromwordtonum(I, d)` — the index of a word, with the empty word $\varnothing \mapsto 0$;
- `fromnumtoword(n, d)` — its inverse;
- `fromlentonum(len_max, d)` — the dimension of the span of all words of length $0, \dots, \text{len\_max}$, namely $\sum_{k=0}^{\text{len\_max}} d^k$.

For $d = 2$ and $\text{len\_max} = 2$ that dimension is $1 + 2 + 4 = 7$, so $L$ will be a $7 \times 7$ matrix. The ordering groups words by increasing length:

In [2]:
d = 2
len_max = 2

for n in range(fromlentonum(len_max, d)):
    word = fromnumtoword(n, d)
    print(f'{n}  ->  {tuple(word) if word else "() empty word"}')

0  ->  () empty word
1  ->  (1,)
2  ->  (2,)
3  ->  (1, 1)
4  ->  (1, 2)
5  ->  (2, 1)
6  ->  (2, 2)




## 1. Building the coefficient table

The table's schema: one row per (`Tensor_rank`, `Component_tensor_Q^rank`, `Order_derivative`, `Derivative_multiindex`), with `Value` the corresponding derivative at $X_0$, the initial value of the polynomial jump-diffusion considered $X$.

Two conventions are worth knowing:

- the $Q^{(r)}$ are **symmetric**, so components are listed only up to permutation — `combinations_with_replacement` generates non-decreasing multi-indices, meaning there is a row for $(1,2)$ but none for $(2,1)$;
- `Order_derivative` is always `len(Derivative_multiindex)`, and ranges from $0$ to `Tensor_rank`.

In [3]:
max_rank = 2

storage = create_storage_df(max_rank, d)
print(f"{len(storage)} coefficient rows for d={d}, max_rank={max_rank}")
storage.head(10)

24 coefficient rows for d=2, max_rank=2


,Tensor_rank,Component_tensor_Q^rank,Order_derivative,Derivative_multiindex,Value
0,1,"(1,)",0,(),1.0
1,1,"(1,)",1,"(1,)",1.0
2,1,"(1,)",1,"(2,)",1.0
3,1,"(2,)",0,(),1.0
4,1,"(2,)",1,"(1,)",1.0
5,1,"(2,)",1,"(2,)",1.0
6,2,"(1, 1)",0,(),1.0
7,2,"(1, 1)",1,"(1,)",1.0
8,2,"(1, 1)",1,"(2,)",1.0
9,2,"(1, 1)",2,"(1, 1)",1.0


## 2. Exporting the table to CSV, fill it in, and reload it

Export the placeholder table to CSV, fill in the model's actual coefficients
by hand (e.g. in a spreadsheet), and save the result.


In [4]:
export_template_df(storage, "coefficients_charact.csv")


Once `coefficients_charact.csv` has been filled in and saved as
`coefficients_charact_filled.csv`, reload it here. (This repo already includes
a filled-in example file so this cell runs out of the box.)

Its entries are mostly the placeholder `1.0` with a handful of arbitrary numbers dropped in, purely so that the matrix assembled at the end is visibly non-trivial. Read the output below as a shape check, not as a model.

In [5]:
storage = load_storage_from_template_df("coefficients_charact_filled.csv")
storage.head(20)


,Tensor_rank,Component_tensor_Q^rank,Order_derivative,Derivative_multiindex,Value
0,1,"(1,)",0,(),1.0
1,1,"(1,)",1,"(1,)",1.0
2,1,"(1,)",1,"(2,)",5.0
3,1,"(2,)",0,(),3.4
4,1,"(2,)",1,"(1,)",5.0
5,1,"(2,)",1,"(2,)",1.0
6,2,"(1, 1)",0,(),1.0
7,2,"(1, 1)",1,"(1,)",1.0
8,2,"(1, 1)",1,"(2,)",1.0
9,2,"(1, 1)",2,"(1, 1)",56.0


## 3. Packing the partial derivatives of each tensor's components into a single flat vector

`derivatives_to_tensor_df` packs the partial derivatives of each
tensor's components, evaluated at $X_0$, into a single flat vector,
indexed according to the word-numbering `fromwordtonum`.

In [6]:
df = derivatives_to_tensor_df(storage, d)
df.head()

,Tensor_rank,Component_tensor_Q^rank,Vector_tensor_num
0,1,"(1,)","[1.0, 1.0, 5.0]"
1,1,"(2,)","[3.4, 5.0, 1.0]"
2,2,"(1, 1)","[1.0, 1.0, 1.0, 56.0, 1.0, 1.0, 1.0]"
3,2,"(1, 2)","[1.0, 1.0, 1.0, 1.0, 2.6, 2.6, 1.0]"
4,2,"(2, 2)","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]"


## 3. Assemble $L$ with `matrix_generator`.

`matrix_generator(coeff_df, len_max, d)` returns the matrix $L$
representing the generator's action on words of length
$0, \dots, \text{len\_max}$, with shape `(dim_L, dim_L)` where
`dim_L = fromlentonum(len_max, d)`.

Specifically, denoting the generator (as an operator) by $L$, and
writing a word $I = (i_1, \dots, i_{|I|})$ of length $|I|$, let
$I^{(k)}$ denote $I$ with its last $k$ letters removed (so
$I' := I^{(1)}$ drops the last letter and $I'' := I^{(2)}$ drops the
last two). The action of $L$ on the basis element $e_I$ is

$$
L e_I = e_{I'} \# \mathbf{b}^{i_{|I|}}
      + \frac{1}{2} e_{I''} \# (\mathbf{a} + \mathbf{k}^2)^{i_{|I|-1}, i_{|I|}}
      + \sum_{k=3}^{|I|} \frac{1}{k!} e_{I^{(k)}} \# (\mathbf{k}^k)^{i_{|I|-k+1}, \dots, i_{|I|}},
$$

where $|I|$ is the lenght of $I$ and $\#$ denotes the shuffle product on the flat word
basis (see `shuffle.py`).

In the vectorial representation, the matrix $L$ returned by
`matrix_generator` satisfies

$$
L \, \mathrm{vec}(e_I) = \mathrm{vec}(L e_I).
$$

### Reading the result

Column $n$ of $L$ holds the image of the basis word `fromnumtoword(n, d)`, expressed in the same basis. Column $0$ is identically zero: the generator annihilates the empty word. 

In [7]:
L = matrix_generator(storage, len_max, d)

dim_L = fromlentonum(len_max, d)
assert L.shape == (dim_L, dim_L)
print("L shape:", L.shape)
L

L shape: (7, 7)


array([[ 0. ,  1. ,  3.4,  0.5,  0.5,  0.5,  0.5],
       [ 0. ,  1. ,  5. ,  1.5,  3.9,  0.5,  0.5],
       [ 0. ,  5. ,  1. ,  0.5,  0.5,  1.5,  3.9],
       [ 0. ,  0. ,  0. , 30. , 10.5,  0.5,  0.5],
       [ 0. ,  0. ,  0. ,  5.5,  2.3,  2.3,  5.5],
       [ 0. ,  0. ,  0. ,  5.5,  2.3,  2.3,  5.5],
       [ 0. ,  0. ,  0. ,  0.5,  0.5, 10.5,  2.5]])